# Analytics Validation: Traceability and Recomputation Checks

This notebook validates the Analytics layer against the persisted tables only. It does not depend on Python variables from `01_build_analytics`. It checks:

1. **Traceability** — Lineage columns non-NULL, single-valued, consistent across tables; natural keys unique; base-value columns present; recorded Gold versions exist in Delta history.
2. **Recomputation** — Every derived metric recomputed from persisted inputs (tolerance 1e-9, null-safe).
3. **Gold reconciliation** — Persisted base values reconciled to Gold at the recorded Delta versions using `VERSION AS OF`.

In [0]:
%python
CATALOG = "tuklas_dev"
ANALYTICS_SCHEMA = "05_analytics"
GOLD_SCHEMA = "04_gold"

BLOCK_COUNT = 0
BLOCK_FAILURES = 0
failures = []
category_counts = {}
REVIEW_COUNT = 0
review_findings = []
gold_versions_ok = False

GOLD_TABLES = ['fact_province_year', 'dim_province', 'dim_year']
ANALYTICS_TABLES = ['tourism_activity', 'tourism_concentration', 'tourism_recovery',
                   'traveler_segment_recovery', 'tourism_economy']

def assert_block(name, condition, details="", category="trace"):
    global BLOCK_COUNT, BLOCK_FAILURES
    BLOCK_COUNT += 1
    if category not in category_counts:
        category_counts[category] = {"count": 0, "failures": 0}
    category_counts[category]["count"] += 1
    if not condition:
        BLOCK_FAILURES += 1
        category_counts[category]["failures"] += 1
        failures.append((name, details, category))
    status = "PASS" if condition else "BLOCK FAIL"
    print(f"  [{status}] {name}: {details}")

def record_review(name, details=""):
    global REVIEW_COUNT
    REVIEW_COUNT += 1
    review_findings.append((name, details))
    print(f"  [REVIEW] {name}: {details}")

print("Configuration complete.")

## Phase 1: Traceability checks

For each of the five Analytics tables, verify:
- All five lineage columns are non-NULL
- Each lineage column has exactly one distinct value per table
- `_analytics_run_id` and Gold versions are identical across all five tables
- Natural keys are present and unique
- Expected base-value columns exist
- Recorded Gold versions exist in Delta history (`DESCRIBE HISTORY` on each Gold table)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_trace_lineage AS
SELECT 'tourism_activity' AS tbl, COUNT(*) AS row_count,
  COUNT(_analytics_run_id) AS nn_run_id, COUNT(DISTINCT _analytics_run_id) AS dist_run_id,
  COUNT(_gold_fact_version) AS nn_fact_v, COUNT(DISTINCT _gold_fact_version) AS dist_fact_v,
  COUNT(_gold_dim_province_version) AS nn_dim_prov_v, COUNT(DISTINCT _gold_dim_province_version) AS dist_dim_prov_v,
  COUNT(_gold_dim_year_version) AS nn_dim_year_v, COUNT(DISTINCT _gold_dim_year_version) AS dist_dim_year_v,
  COUNT(_processed_at) AS nn_processed_at, COUNT(DISTINCT _processed_at) AS dist_processed_at
FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
UNION ALL
SELECT 'tourism_concentration', COUNT(*),
  COUNT(_analytics_run_id), COUNT(DISTINCT _analytics_run_id),
  COUNT(_gold_fact_version), COUNT(DISTINCT _gold_fact_version),
  COUNT(_gold_dim_province_version), COUNT(DISTINCT _gold_dim_province_version),
  COUNT(_gold_dim_year_version), COUNT(DISTINCT _gold_dim_year_version),
  COUNT(_processed_at), COUNT(DISTINCT _processed_at)
FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`
UNION ALL
SELECT 'tourism_recovery', COUNT(*),
  COUNT(_analytics_run_id), COUNT(DISTINCT _analytics_run_id),
  COUNT(_gold_fact_version), COUNT(DISTINCT _gold_fact_version),
  COUNT(_gold_dim_province_version), COUNT(DISTINCT _gold_dim_province_version),
  COUNT(_gold_dim_year_version), COUNT(DISTINCT _gold_dim_year_version),
  COUNT(_processed_at), COUNT(DISTINCT _processed_at)
FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`
UNION ALL
SELECT 'traveler_segment_recovery', COUNT(*),
  COUNT(_analytics_run_id), COUNT(DISTINCT _analytics_run_id),
  COUNT(_gold_fact_version), COUNT(DISTINCT _gold_fact_version),
  COUNT(_gold_dim_province_version), COUNT(DISTINCT _gold_dim_province_version),
  COUNT(_gold_dim_year_version), COUNT(DISTINCT _gold_dim_year_version),
  COUNT(_processed_at), COUNT(DISTINCT _processed_at)
FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
UNION ALL
SELECT 'tourism_economy', COUNT(*),
  COUNT(_analytics_run_id), COUNT(DISTINCT _analytics_run_id),
  COUNT(_gold_fact_version), COUNT(DISTINCT _gold_fact_version),
  COUNT(_gold_dim_province_version), COUNT(DISTINCT _gold_dim_province_version),
  COUNT(_gold_dim_year_version), COUNT(DISTINCT _gold_dim_year_version),
  COUNT(_processed_at), COUNT(DISTINCT _processed_at)
FROM `tuklas_dev`.`05_analytics`.`tourism_economy`

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_trace_cross AS
WITH run_ids AS (
  SELECT DISTINCT _analytics_run_id AS val FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
  UNION SELECT DISTINCT _analytics_run_id FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`
  UNION SELECT DISTINCT _analytics_run_id FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`
  UNION SELECT DISTINCT _analytics_run_id FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
  UNION SELECT DISTINCT _analytics_run_id FROM `tuklas_dev`.`05_analytics`.`tourism_economy`
),
fact_versions AS (
  SELECT DISTINCT _gold_fact_version AS val FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
  UNION SELECT DISTINCT _gold_fact_version FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`
  UNION SELECT DISTINCT _gold_fact_version FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`
  UNION SELECT DISTINCT _gold_fact_version FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
  UNION SELECT DISTINCT _gold_fact_version FROM `tuklas_dev`.`05_analytics`.`tourism_economy`
),
dim_prov_versions AS (
  SELECT DISTINCT _gold_dim_province_version AS val FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
  UNION SELECT DISTINCT _gold_dim_province_version FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`
  UNION SELECT DISTINCT _gold_dim_province_version FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`
  UNION SELECT DISTINCT _gold_dim_province_version FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
  UNION SELECT DISTINCT _gold_dim_province_version FROM `tuklas_dev`.`05_analytics`.`tourism_economy`
),
dim_year_versions AS (
  SELECT DISTINCT _gold_dim_year_version AS val FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
  UNION SELECT DISTINCT _gold_dim_year_version FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`
  UNION SELECT DISTINCT _gold_dim_year_version FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`
  UNION SELECT DISTINCT _gold_dim_year_version FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
  UNION SELECT DISTINCT _gold_dim_year_version FROM `tuklas_dev`.`05_analytics`.`tourism_economy`
)
SELECT
  (SELECT COUNT(*) FROM run_ids) AS distinct_run_ids,
  (SELECT COUNT(*) FROM fact_versions) AS distinct_fact_versions,
  (SELECT COUNT(*) FROM dim_prov_versions) AS distinct_dim_prov_versions,
  (SELECT COUNT(*) FROM dim_year_versions) AS distinct_dim_year_versions

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_trace_keys AS
SELECT 'tourism_activity' AS tbl, COUNT(*) AS row_count, COUNT(DISTINCT province_psgc, year) AS distinct_keys
FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
UNION ALL
SELECT 'tourism_concentration', COUNT(*), COUNT(DISTINCT year)
FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`
UNION ALL
SELECT 'tourism_recovery', COUNT(*), COUNT(DISTINCT province_psgc, baseline_year, comparison_year)
FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`
UNION ALL
SELECT 'traveler_segment_recovery', COUNT(*), COUNT(DISTINCT province_psgc, baseline_year, comparison_year)
FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
UNION ALL
SELECT 'tourism_economy', COUNT(*), COUNT(DISTINCT province_psgc, baseline_year, comparison_year)
FROM `tuklas_dev`.`05_analytics`.`tourism_economy`

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_trace_cols AS
WITH expected AS (
  SELECT 'tourism_activity' AS tbl, array(
    'province_psgc','province_name','unit_type','region_psgc','region_name','year',
    'total_travelers','population_total','population_source',
    'traveler_value_status','comparable_to_2019','comparability_note','foreign_includes_overseas_filipinos',
    'derived_values','per_capita_recomputed',
    'activity_eligible','activity_exclusion_reason','intensity_eligible','intensity_exclusion_reason',
    'tourism_intensity_per_1000','tourism_rank','intensity_rank','tourism_share_pct'
  ) AS cols
  UNION SELECT 'tourism_concentration', array(
    'year','eligible_reporting_unit_count','excluded_reporting_unit_count',
    'eligible_total_travelers','top_5_travelers','top_5_unit_count','top_5_share_pct',
    'top_10_travelers','top_10_unit_count','top_10_share_pct'
  )
  UNION SELECT 'tourism_recovery', array(
    'province_psgc','province_name','unit_type','region_psgc','region_name','comparability_note',
    'baseline_year','comparison_year',
    'travelers_2019','travelers_comparison_year',
    'status_2019','status_comparison','comp_2019','comp_comparison',
    'foreign_includes_overseas_filipinos',
    'derived_values_2019','derived_values_comparison','per_capita_recomputed_2019','per_capita_recomputed_comparison',
    'recovery_eligible','recovery_exclusion_reason',
    'traveler_change_absolute','recovery_index','recovery_class'
  )
  UNION SELECT 'traveler_segment_recovery', array(
    'province_psgc','province_name','unit_type','region_psgc','region_name','comparability_note',
    'baseline_year','comparison_year',
    'below_2019','overall_recovery_index','overall_recovery_class',
    'travelers_2019','travelers_2023','comparable_to_2019',
    'foreign_includes_overseas_filipinos',
    'derived_values_2019','per_capita_recomputed_2019','derived_values_2023','per_capita_recomputed_2023',
    'domestic_travelers_2019','domestic_travelers_2023','domestic_recovery_index','domestic_recovery_class',
    'domestic_eligible','domestic_exclusion_reason',
    'foreign_travelers_2019','foreign_travelers_2023','foreign_recovery_index','foreign_recovery_class',
    'foreign_eligible','foreign_exclusion_reason',
    'overseas_filipino_travelers_2019','overseas_filipino_travelers_2023',
    'overseas_filipino_recovery_index','overseas_filipino_recovery_class',
    'overseas_filipino_eligible','overseas_filipino_exclusion_reason'
  )
  UNION SELECT 'tourism_economy', array(
    'province_psgc','province_name','unit_type','region_psgc','region_name',
    'baseline_year','comparison_year',
    'travelers_2019','travelers_2023',
    'status_2019','status_2023','comp_2019','comp_2023',
    'recovery_index_2023','recovery_class_2023','recovery_eligible_2023','recovery_exclusion_reason_2023',
    'afs_gva_constant_2019','afs_gva_constant_2023','afs_gva_current_2023','gdp_total_current_2023',
    'gdp_per_capita_current_2019','gdp_per_capita_current_2023',
    'population_total_2023','population_source_2023',
    'derived_values_2019','derived_values_2023','per_capita_recomputed_2019','per_capita_recomputed_2023',
    'tourism_change_eligible','afs_growth_eligible','afs_share_eligible','tourism_intensity_eligible','alignment_eligible',
    'tourism_change_pct','afs_gva_growth_pct','tourism_direction','economy_direction',
    'tourism_economy_pattern','tourism_economy_pattern_label',
    'afs_share_of_gdp_pct','tourism_intensity_per_1000_2023',
    'tourism_change_exclusion_reason','afs_growth_exclusion_reason','afs_share_exclusion_reason',
    'tourism_intensity_exclusion_reason','alignment_exclusion_reason'
  )
),
actual AS (
  SELECT table_name, collect_set(column_name) AS present
  FROM `tuklas_dev`.information_schema.columns
  WHERE table_schema = '05_analytics'
  GROUP BY table_name
)
SELECT e.tbl, SIZE(array_except(e.cols, a.present)) AS missing_count, e.cols AS expected_cols, a.present AS actual_cols
FROM expected e JOIN actual a ON e.tbl = a.table_name

In [0]:
%python
# Lineage non-NULL and single-value
lineage = spark.sql("SELECT * FROM v_trace_lineage").collect()
for row in lineage:
    tbl = row['tbl']
    rc = row['row_count']
    assert_block(f"lineage_nn_{tbl}_run_id", row['nn_run_id'] == rc, f"NULLs in _analytics_run_id ({row['nn_run_id']}/{rc})", "trace")
    assert_block(f"lineage_distinct_{tbl}_run_id", row['dist_run_id'] == 1, f"{row['dist_run_id']} distinct run_ids", "trace")
    assert_block(f"lineage_nn_{tbl}_fact_v", row['nn_fact_v'] == rc, f"NULLs in _gold_fact_version", "trace")
    assert_block(f"lineage_distinct_{tbl}_fact_v", row['dist_fact_v'] == 1, f"{row['dist_fact_v']} distinct fact versions", "trace")
    assert_block(f"lineage_nn_{tbl}_dim_prov_v", row['nn_dim_prov_v'] == rc, f"NULLs in _gold_dim_province_version", "trace")
    assert_block(f"lineage_distinct_{tbl}_dim_prov_v", row['dist_dim_prov_v'] == 1, f"{row['dist_dim_prov_v']} distinct dim_prov versions", "trace")
    assert_block(f"lineage_nn_{tbl}_dim_year_v", row['nn_dim_year_v'] == rc, f"NULLs in _gold_dim_year_version", "trace")
    assert_block(f"lineage_distinct_{tbl}_dim_year_v", row['dist_dim_year_v'] == 1, f"{row['dist_dim_year_v']} distinct dim_year versions", "trace")
    assert_block(f"lineage_nn_{tbl}_processed_at", row['nn_processed_at'] == rc, f"NULLs in _processed_at", "trace")
    assert_block(f"lineage_distinct_{tbl}_processed_at", row['dist_processed_at'] == 1, f"{row['dist_processed_at']} distinct processed_at", "trace")

# Cross-table consistency
cross = spark.sql("SELECT * FROM v_trace_cross").first()
assert_block("cross_run_id", cross['distinct_run_ids'] == 1, f"{cross['distinct_run_ids']} distinct run_ids across tables", "trace")
assert_block("cross_fact_version", cross['distinct_fact_versions'] == 1, f"{cross['distinct_fact_versions']} distinct fact versions", "trace")
assert_block("cross_dim_prov_version", cross['distinct_dim_prov_versions'] == 1, f"{cross['distinct_dim_prov_versions']} distinct dim_prov versions", "trace")
assert_block("cross_dim_year_version", cross['distinct_dim_year_versions'] == 1, f"{cross['distinct_dim_year_versions']} distinct dim_year versions", "trace")

# _processed_at identical across all five tables
processed_at_counts = spark.sql("""
  SELECT COUNT(*) AS cnt FROM (
    SELECT DISTINCT _processed_at FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
    UNION SELECT DISTINCT _processed_at FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`
    UNION SELECT DISTINCT _processed_at FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`
    UNION SELECT DISTINCT _processed_at FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
    UNION SELECT DISTINCT _processed_at FROM `tuklas_dev`.`05_analytics`.`tourism_economy`
  )
""").first()['cnt']
assert_block("cross_processed_at", processed_at_counts == 1, f"{processed_at_counts} distinct _processed_at values across tables", "trace")

# Natural keys
keys = spark.sql("SELECT * FROM v_trace_keys").collect()
for row in keys:
    tbl = row['tbl']
    assert_block(f"key_unique_{tbl}", row['row_count'] == row['distinct_keys'],
        f"row_count={row['row_count']}, distinct_keys={row['distinct_keys']}", "trace")

# Base-value columns — assert exactly 5 tables returned
cols = spark.sql("SELECT * FROM v_trace_cols").collect()
assert_block("base_cols_5_tables", len(cols) == 5, f"{len(cols)} tables returned (expected 5)", "trace")
for row in cols:
    tbl = row['tbl']
    assert_block(f"base_cols_{tbl}", row['missing_count'] == 0, f"{row['missing_count']} missing columns in {tbl}", "trace")

# Row-count checks
assert_block("rowcount_tourism_activity",
    spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_activity`").first()[0] == 600,
    "expected 600", "rowcount")
assert_block("rowcount_tourism_concentration",
    spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`").first()[0] == 6,
    "expected 6", "rowcount")
assert_block("rowcount_tourism_recovery",
    spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`").first()[0] == 200,
    "expected 200", "rowcount")
assert_block("rowcount_tourism_economy",
    spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_economy`").first()[0] == 100,
    "expected 100", "rowcount")
cohort_count = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_recovery` WHERE comparison_year = 2023 AND recovery_eligible = true AND recovery_index < 100").first()[0]
segment_count = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`").first()[0]
assert_block("rowcount_segment_matches_cohort", segment_count == cohort_count,
    f"segment={segment_count}, cohort={cohort_count}", "rowcount")

# Gold version existence — DESCRIBE HISTORY + .collect() + REVIEW if current is newer
run_id_row = spark.sql("SELECT _analytics_run_id, _gold_fact_version, _gold_dim_province_version, _gold_dim_year_version FROM `tuklas_dev`.`05_analytics`.`tourism_activity` LIMIT 1").first()
stored_run_id = run_id_row['_analytics_run_id']
stored_fact_v = int(run_id_row['_gold_fact_version'])
stored_dim_prov_v = int(run_id_row['_gold_dim_province_version'])
stored_dim_year_v = int(run_id_row['_gold_dim_year_version'])

gold_versions_ok = True
for tbl, v in [('fact_province_year', stored_fact_v), ('dim_province', stored_dim_prov_v), ('dim_year', stored_dim_year_v)]:
    # (a) confirm version appears in DESCRIBE HISTORY
    in_history = spark.sql(f"DESCRIBE HISTORY `tuklas_dev`.`04_gold`.`{tbl}`").filter(f"version = {v}").count() > 0
    assert_block(f"gold_version_in_history_{tbl}", in_history, f"Version {v} in DESCRIBE HISTORY", "trace")
    if not in_history:
        gold_versions_ok = False
    # (b) execute a .collect() read to detect vacuumed versions
    try:
        spark.sql(f"SELECT * FROM `tuklas_dev`.`04_gold`.`{tbl}` VERSION AS OF {v} LIMIT 1").collect()
        assert_block(f"gold_version_readable_{tbl}", True, f"Version {v} readable via .collect()", "trace")
    except Exception as e:
        assert_block(f"gold_version_readable_{tbl}", False, f"Version {v} not readable: {e}", "trace")
        gold_versions_ok = False
    # (c) REVIEW if current Gold version is newer than stored
    current_v = spark.sql(f"DESCRIBE HISTORY `tuklas_dev`.`04_gold`.`{tbl}`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
    if int(current_v) > v:
        record_review(f"gold_version_newer_{tbl}", f"Stored v{v}, current v{current_v} — Gold was updated after Analytics build")

print(f"\nTraceability checks: {BLOCK_COUNT} BLOCK checks, {BLOCK_FAILURES} failures, {REVIEW_COUNT} REVIEW findings")
if not gold_versions_ok:
    print("Gold version checks failed — Phase 3 will be skipped.")

## Phase 2: Recomputation checks

Recompute every derived metric from persisted inputs (tolerance 1e-9, null-safe). Mismatches are BLOCK failures.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_recovery AS
SELECT province_psgc, baseline_year, comparison_year,
  recovery_eligible AS stored_eligible,
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0
       AND status_2019 != 'not_reported'
       AND comp_2019 IS NOT NULL AND comp_2019 = true
       AND travelers_comparison_year IS NOT NULL AND travelers_comparison_year >= 0
       AND status_comparison != 'not_reported'
       AND comp_comparison IS NOT NULL AND comp_comparison = true
       THEN true ELSE false END AS recomputed_eligible,
  recovery_index AS stored_index,
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0
       AND status_2019 != 'not_reported'
       AND comp_2019 IS NOT NULL AND comp_2019 = true
       AND travelers_comparison_year IS NOT NULL AND travelers_comparison_year >= 0
       AND status_comparison != 'not_reported'
       AND comp_comparison IS NOT NULL AND comp_comparison = true
       THEN CAST(travelers_comparison_year AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_index,
  recovery_class AS stored_class,
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0
       AND status_2019 != 'not_reported'
       AND comp_2019 IS NOT NULL AND comp_2019 = true
       AND travelers_comparison_year IS NOT NULL AND travelers_comparison_year >= 0
       AND status_comparison != 'not_reported'
       AND comp_comparison IS NOT NULL AND comp_comparison = true
       THEN CASE WHEN CAST(travelers_comparison_year AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(travelers_comparison_year AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS recomputed_class,
  traveler_change_absolute AS stored_change_absolute,
  (travelers_comparison_year - travelers_2019) AS recomputed_change_absolute
FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_intensity AS
SELECT province_psgc, year,
  intensity_eligible AS stored_eligible,
  CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
            AND population_total IS NOT NULL AND population_total > 0
       THEN true ELSE false END AS recomputed_eligible,
  tourism_intensity_per_1000 AS stored_val,
  CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
            AND population_total IS NOT NULL AND population_total > 0
       THEN CAST(total_travelers AS DOUBLE) * 1000.0 / CAST(population_total AS DOUBLE)
       ELSE NULL END AS recomputed_val
FROM `tuklas_dev`.`05_analytics`.`tourism_activity`

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_share AS
WITH eligible_totals AS (
  SELECT year, SUM(total_travelers) AS eligible_total
  FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
  WHERE traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
  GROUP BY year
)
SELECT a.province_psgc, a.year,
  a.activity_eligible AS stored_eligible,
  CASE WHEN a.traveler_value_status != 'not_reported' AND a.total_travelers IS NOT NULL
       THEN true ELSE false END AS recomputed_eligible,
  a.tourism_share_pct AS stored_val,
  CASE WHEN a.traveler_value_status != 'not_reported' AND a.total_travelers IS NOT NULL
       THEN CAST(a.total_travelers AS DOUBLE) * 100.0 / e.eligible_total
       ELSE NULL END AS recomputed_val
FROM `tuklas_dev`.`05_analytics`.`tourism_activity` a
JOIN eligible_totals e ON a.year = e.year

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_concentration AS
SELECT year,
  top_5_share_pct AS stored_5,
  CASE WHEN eligible_total_travelers > 0
       THEN CAST(top_5_travelers AS DOUBLE) * 100.0 / eligible_total_travelers
       ELSE NULL END AS recomputed_5,
  top_10_share_pct AS stored_10,
  CASE WHEN eligible_total_travelers > 0
       THEN CAST(top_10_travelers AS DOUBLE) * 100.0 / eligible_total_travelers
       ELSE NULL END AS recomputed_10
FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_economy AS
WITH base AS (
  SELECT e.*,
    CASE WHEN e.travelers_2019 IS NOT NULL AND e.travelers_2019 > 0 AND e.status_2019 != 'not_reported' AND e.comp_2019 IS NOT NULL AND e.comp_2019 = true AND e.travelers_2023 IS NOT NULL AND e.travelers_2023 >= 0 AND e.status_2023 != 'not_reported' AND e.comp_2023 IS NOT NULL AND e.comp_2023 = true THEN true ELSE false END AS recomp_tourism_change_eligible,
    CASE WHEN e.afs_gva_constant_2019 IS NOT NULL AND e.afs_gva_constant_2023 IS NOT NULL AND e.afs_gva_constant_2019 > 0 THEN true ELSE false END AS recomp_afs_growth_eligible,
    CASE WHEN e.afs_gva_current_2023 IS NOT NULL AND e.gdp_total_current_2023 IS NOT NULL AND e.gdp_total_current_2023 > 0 THEN true ELSE false END AS recomp_afs_share_eligible,
    CASE WHEN e.travelers_2023 IS NOT NULL AND e.population_total_2023 IS NOT NULL AND e.population_total_2023 > 0 THEN true ELSE false END AS recomp_tourism_intensity_eligible,
    CASE WHEN e.travelers_2019 IS NOT NULL AND e.travelers_2019 > 0 AND e.status_2019 != 'not_reported' AND e.comp_2019 IS NOT NULL AND e.comp_2019 = true AND e.travelers_2023 IS NOT NULL AND e.travelers_2023 >= 0 AND e.status_2023 != 'not_reported' AND e.comp_2023 IS NOT NULL AND e.comp_2023 = true AND e.afs_gva_constant_2019 IS NOT NULL AND e.afs_gva_constant_2023 IS NOT NULL AND e.afs_gva_constant_2019 > 0 THEN true ELSE false END AS recomp_alignment_eligible
  FROM `tuklas_dev`.`05_analytics`.`tourism_economy` e
)
SELECT province_psgc,
  tourism_change_eligible AS stored_tourism_change_eligible,
  afs_growth_eligible AS stored_afs_growth_eligible,
  afs_share_eligible AS stored_afs_share_eligible,
  tourism_intensity_eligible AS stored_tourism_intensity_eligible,
  alignment_eligible AS stored_alignment_eligible,
  recomp_tourism_change_eligible,
  recomp_afs_growth_eligible,
  recomp_afs_share_eligible,
  recomp_tourism_intensity_eligible,
  recomp_alignment_eligible,
  tourism_change_pct AS stored_change,
  CASE WHEN recomp_tourism_change_eligible
       THEN (CAST(travelers_2023 AS DOUBLE) - CAST(travelers_2019 AS DOUBLE)) * 100.0 / CAST(travelers_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_change,
  afs_gva_growth_pct AS stored_growth,
  CASE WHEN recomp_afs_growth_eligible
       THEN (CAST(afs_gva_constant_2023 AS DOUBLE) - CAST(afs_gva_constant_2019 AS DOUBLE)) * 100.0 / CAST(afs_gva_constant_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_growth,
  afs_share_of_gdp_pct AS stored_share,
  CASE WHEN recomp_afs_share_eligible
       THEN CAST(afs_gva_current_2023 AS DOUBLE) * 100.0 / CAST(gdp_total_current_2023 AS DOUBLE)
       ELSE NULL END AS recomputed_share,
  tourism_intensity_per_1000_2023 AS stored_intensity,
  CASE WHEN recomp_tourism_intensity_eligible
       THEN CAST(travelers_2023 AS DOUBLE) * 1000.0 / CAST(population_total_2023 AS DOUBLE)
       ELSE NULL END AS recomputed_intensity,
  tourism_direction AS stored_tourism_dir,
  CASE WHEN recomp_tourism_change_eligible THEN
       CASE WHEN travelers_2023 > travelers_2019 THEN 'up' WHEN travelers_2023 = travelers_2019 THEN 'unchanged' ELSE 'down' END
       ELSE NULL END AS recomputed_tourism_dir,
  economy_direction AS stored_econ_dir,
  CASE WHEN recomp_afs_growth_eligible THEN
       CASE WHEN afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'up' WHEN afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'unchanged' ELSE 'down' END
       ELSE NULL END AS recomputed_econ_dir,
  tourism_economy_pattern AS stored_pattern,
  CASE WHEN recomp_alignment_eligible THEN
    CASE WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_up_economy_up'
         WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_up_economy_down'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_down_economy_up'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_down_economy_down'
         WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_unchanged_economy_up'
         WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_unchanged_economy_down'
         WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'tourism_up_economy_unchanged'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'tourism_down_economy_unchanged'
         ELSE 'tourism_unchanged_economy_unchanged' END
    ELSE NULL END AS recomputed_pattern,
  recovery_eligible_2023 AS stored_recovery_eligible_2023,
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0
            AND status_2019 != 'not_reported'
            AND comp_2019 IS NOT NULL AND comp_2019 = true
            AND travelers_2023 IS NOT NULL AND travelers_2023 >= 0
            AND status_2023 != 'not_reported'
            AND comp_2023 IS NOT NULL AND comp_2023 = true
       THEN true ELSE false END AS recomputed_recovery_eligible_2023,
  recovery_index_2023 AS stored_recovery_index_2023,
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0
            AND status_2019 != 'not_reported'
            AND comp_2019 IS NOT NULL AND comp_2019 = true
            AND travelers_2023 IS NOT NULL AND travelers_2023 >= 0
            AND status_2023 != 'not_reported'
            AND comp_2023 IS NOT NULL AND comp_2023 = true
       THEN CAST(travelers_2023 AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_recovery_index_2023,
  recovery_class_2023 AS stored_recovery_class_2023,
  CASE WHEN travelers_2019 IS NOT NULL AND travelers_2019 > 0
            AND status_2019 != 'not_reported'
            AND comp_2019 IS NOT NULL AND comp_2019 = true
            AND travelers_2023 IS NOT NULL AND travelers_2023 >= 0
            AND status_2023 != 'not_reported'
            AND comp_2023 IS NOT NULL AND comp_2023 = true
       THEN CASE WHEN CAST(travelers_2023 AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(travelers_2023 AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS recomputed_recovery_class_2023,
  tourism_economy_pattern_label AS stored_pattern_label,
  CASE WHEN recomp_alignment_eligible
   AND (
     CASE WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_up_economy_up'
         WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_up_economy_down'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_down_economy_up'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_down_economy_down'
         WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_unchanged_economy_up'
         WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_unchanged_economy_down'
         WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'tourism_up_economy_unchanged'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'tourism_down_economy_unchanged'
         ELSE 'tourism_unchanged_economy_unchanged' END
   ) IS NOT NULL THEN
     CASE WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'Tourism activity increased; real A&F GVA increased'
          WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'Tourism activity increased; real A&F GVA decreased'
          WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'Tourism activity decreased; real A&F GVA increased'
          WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'Tourism activity decreased; real A&F GVA decreased'
          WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'Tourism activity unchanged; real A&F GVA increased'
          WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'Tourism activity unchanged; real A&F GVA decreased'
          WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'Tourism activity increased; real A&F GVA unchanged'
          WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'Tourism activity decreased; real A&F GVA unchanged'
          ELSE 'Tourism activity unchanged; real A&F GVA unchanged' END
   ELSE NULL END AS recomputed_pattern_label
FROM base

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_segment AS
WITH base AS (
  SELECT s.*,
    CASE WHEN s.domestic_travelers_2019 IS NOT NULL AND s.domestic_travelers_2019 > 0 AND s.domestic_travelers_2023 IS NOT NULL AND s.domestic_travelers_2023 >= 0 THEN true ELSE false END AS recomp_domestic_eligible,
    CASE WHEN COALESCE(s.foreign_includes_overseas_filipinos, false) = false AND s.foreign_travelers_2019 IS NOT NULL AND s.foreign_travelers_2019 > 0 AND s.foreign_travelers_2023 IS NOT NULL AND s.foreign_travelers_2023 >= 0 THEN true ELSE false END AS recomp_foreign_eligible,
    CASE WHEN COALESCE(s.foreign_includes_overseas_filipinos, false) = false AND s.overseas_filipino_travelers_2019 IS NOT NULL AND s.overseas_filipino_travelers_2019 > 0 AND s.overseas_filipino_travelers_2023 IS NOT NULL AND s.overseas_filipino_travelers_2023 >= 0 THEN true ELSE false END AS recomp_overseas_filipino_eligible,
    CASE WHEN s.travelers_2019 IS NOT NULL AND s.travelers_2019 > 0 AND s.travelers_2023 IS NOT NULL AND s.travelers_2023 >= 0 THEN true ELSE false END AS recomp_overall_eligible
  FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` s
)
SELECT province_psgc,
  domestic_eligible AS stored_domestic_eligible,
  foreign_eligible AS stored_foreign_eligible,
  overseas_filipino_eligible AS stored_overseas_filipino_eligible,
  recomp_domestic_eligible,
  recomp_foreign_eligible,
  recomp_overseas_filipino_eligible,
  domestic_recovery_index AS stored_dom_idx,
  CASE WHEN recomp_domestic_eligible AND domestic_travelers_2019 > 0
       THEN CAST(domestic_travelers_2023 AS DOUBLE) * 100.0 / CAST(domestic_travelers_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_dom_idx,
  domestic_recovery_class AS stored_dom_class,
  CASE WHEN recomp_domestic_eligible AND domestic_travelers_2019 > 0
       THEN CASE WHEN CAST(domestic_travelers_2023 AS DOUBLE) * 100.0 / CAST(domestic_travelers_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(domestic_travelers_2023 AS DOUBLE) * 100.0 / CAST(domestic_travelers_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS recomputed_dom_class,
  foreign_recovery_index AS stored_for_idx,
  CASE WHEN recomp_foreign_eligible AND foreign_travelers_2019 > 0
       THEN CAST(foreign_travelers_2023 AS DOUBLE) * 100.0 / CAST(foreign_travelers_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_for_idx,
  foreign_recovery_class AS stored_for_class,
  CASE WHEN recomp_foreign_eligible AND foreign_travelers_2019 > 0
       THEN CASE WHEN CAST(foreign_travelers_2023 AS DOUBLE) * 100.0 / CAST(foreign_travelers_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(foreign_travelers_2023 AS DOUBLE) * 100.0 / CAST(foreign_travelers_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS recomputed_for_class,
  overseas_filipino_recovery_index AS stored_of_idx,
  CASE WHEN recomp_overseas_filipino_eligible AND overseas_filipino_travelers_2019 > 0
       THEN CAST(overseas_filipino_travelers_2023 AS DOUBLE) * 100.0 / CAST(overseas_filipino_travelers_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_of_idx,
  overseas_filipino_recovery_class AS stored_of_class,
  CASE WHEN recomp_overseas_filipino_eligible AND overseas_filipino_travelers_2019 > 0
       THEN CASE WHEN CAST(overseas_filipino_travelers_2023 AS DOUBLE) * 100.0 / CAST(overseas_filipino_travelers_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(overseas_filipino_travelers_2023 AS DOUBLE) * 100.0 / CAST(overseas_filipino_travelers_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS recomputed_of_class,
  overall_recovery_index AS stored_overall_idx,
  CASE WHEN recomp_overall_eligible AND travelers_2019 > 0
       THEN CAST(travelers_2023 AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE)
       ELSE NULL END AS recomputed_overall_idx,
  overall_recovery_class AS stored_overall_class,
  CASE WHEN recomp_overall_eligible AND travelers_2019 > 0
       THEN CASE WHEN CAST(travelers_2023 AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(travelers_2023 AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS recomputed_overall_class
FROM base

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_ranks AS
SELECT province_psgc, year,
  tourism_rank AS stored_tourism_rank,
  CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
       THEN RANK() OVER (
         PARTITION BY year, CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL THEN 1 ELSE 0 END
         ORDER BY total_travelers DESC, province_psgc ASC)
       ELSE NULL END AS recomputed_tourism_rank,
  intensity_rank AS stored_intensity_rank,
  CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
            AND population_total IS NOT NULL AND population_total > 0
       THEN RANK() OVER (
         PARTITION BY year, CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL AND population_total IS NOT NULL AND population_total > 0 THEN 1 ELSE 0 END
         ORDER BY CAST(total_travelers AS DOUBLE) * 1000.0 / CAST(population_total AS DOUBLE) DESC, province_psgc ASC)
       ELSE NULL END AS recomputed_intensity_rank
FROM `tuklas_dev`.`05_analytics`.`tourism_activity`

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_recomp_concentration_indep AS
WITH eligible AS (
  SELECT year, province_psgc, total_travelers
  FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
  WHERE traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
),
ranked AS (
  SELECT year, province_psgc, total_travelers,
    ROW_NUMBER() OVER (PARTITION BY year ORDER BY total_travelers DESC, province_psgc ASC) AS rn
  FROM eligible
),
year_totals AS (
  SELECT year, COUNT(*) AS eligible_count, SUM(total_travelers) AS eligible_total
  FROM eligible GROUP BY year
),
total_units AS (
  SELECT year, COUNT(*) AS total_count
  FROM `tuklas_dev`.`05_analytics`.`tourism_activity` GROUP BY year
),
top5 AS (
  SELECT year, SUM(total_travelers) AS top_5_travelers, COUNT(*) AS top_5_count
  FROM ranked WHERE rn <= 5 GROUP BY year
),
top10 AS (
  SELECT year, SUM(total_travelers) AS top_10_travelers, COUNT(*) AS top_10_count
  FROM ranked WHERE rn <= 10 GROUP BY year
),
stored AS (
  SELECT c.year, c.eligible_reporting_unit_count, c.excluded_reporting_unit_count,
    c.eligible_total_travelers, c.top_5_unit_count, c.top_10_unit_count,
    c.top_5_travelers, c.top_10_travelers, c.top_5_share_pct, c.top_10_share_pct
  FROM `tuklas_dev`.`05_analytics`.`tourism_concentration` c
)
SELECT s.year,
  s.eligible_reporting_unit_count AS stored_eligible_count,
  yt.eligible_count AS recomputed_eligible_count,
  s.eligible_total_travelers AS stored_eligible_total,
  yt.eligible_total AS recomputed_eligible_total,
  s.top_5_travelers AS stored_top_5,
  COALESCE(t5.top_5_travelers, 0) AS recomputed_top_5,
  s.top_10_travelers AS stored_top_10,
  COALESCE(t10.top_10_travelers, 0) AS recomputed_top_10,
  s.top_5_share_pct AS stored_top_5_share,
  CASE WHEN yt.eligible_total > 0 THEN COALESCE(t5.top_5_travelers, 0) * 100.0 / yt.eligible_total ELSE NULL END AS recomputed_top_5_share,
  s.top_10_share_pct AS stored_top_10_share,
  CASE WHEN yt.eligible_total > 0 THEN COALESCE(t10.top_10_travelers, 0) * 100.0 / yt.eligible_total ELSE NULL END AS recomputed_top_10_share,
  s.excluded_reporting_unit_count AS stored_excluded_count,
  COALESCE(tu.total_count - yt.eligible_count, 0) AS recomputed_excluded_count,
  s.top_5_unit_count AS stored_top_5_unit_count,
  COALESCE(t5.top_5_count, 0) AS recomputed_top_5_unit_count,
  s.top_10_unit_count AS stored_top_10_unit_count,
  COALESCE(t10.top_10_count, 0) AS recomputed_top_10_unit_count
FROM stored s
JOIN year_totals yt ON s.year = yt.year
LEFT JOIN total_units tu ON s.year = tu.year
LEFT JOIN top5 t5 ON s.year = t5.year
LEFT JOIN top10 t10 ON s.year = t10.year

In [0]:
%python
def check_recomp_num(name, view, stored_col, recomputed_col, category="recomp"):
    mismatches = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {view}
        WHERE NOT (
          ({stored_col} IS NULL AND {recomputed_col} IS NULL)
          OR ({stored_col} IS NOT NULL AND {recomputed_col} IS NOT NULL
              AND ABS(CAST({stored_col} AS DOUBLE) - CAST({recomputed_col} AS DOUBLE)) <= 1e-9)
        )
    """).first()['cnt']
    assert_block(name, mismatches == 0, f"{mismatches} mismatches", category)

def check_recomp_eq(name, view, stored_col, recomputed_col, category="recomp"):
    mismatches = spark.sql(f"""
        SELECT COUNT(*) AS cnt FROM {view}
        WHERE NOT ({stored_col} <=> {recomputed_col})
    """).first()['cnt']
    assert_block(name, mismatches == 0, f"{mismatches} mismatches", category)

# Recovery
check_recomp_eq("recomp_recovery_eligible", "v_recomp_recovery", "stored_eligible", "recomputed_eligible")
check_recomp_num("recomp_recovery_index", "v_recomp_recovery", "stored_index", "recomputed_index")
check_recomp_eq("recomp_recovery_class", "v_recomp_recovery", "stored_class", "recomputed_class")
check_recomp_num("recomp_traveler_change_absolute", "v_recomp_recovery", "stored_change_absolute", "recomputed_change_absolute")

# Intensity
check_recomp_eq("recomp_intensity_eligible", "v_recomp_intensity", "stored_eligible", "recomputed_eligible")
check_recomp_num("recomp_intensity", "v_recomp_intensity", "stored_val", "recomputed_val")

# Share
check_recomp_eq("recomp_share_eligible", "v_recomp_share", "stored_eligible", "recomputed_eligible")
check_recomp_num("recomp_share", "v_recomp_share", "stored_val", "recomputed_val")

# Ranks
check_recomp_eq("recomp_tourism_rank", "v_recomp_ranks", "stored_tourism_rank", "recomputed_tourism_rank")
check_recomp_eq("recomp_intensity_rank", "v_recomp_ranks", "stored_intensity_rank", "recomputed_intensity_rank")

# Concentration (from stored sums)
check_recomp_num("recomp_top5_share", "v_recomp_concentration", "stored_5", "recomputed_5")
check_recomp_num("recomp_top10_share", "v_recomp_concentration", "stored_10", "recomputed_10")

# Concentration (independent from tourism_activity)
check_recomp_num("recomp_indep_eligible_count", "v_recomp_concentration_indep", "stored_eligible_count", "recomputed_eligible_count")
check_recomp_num("recomp_indep_eligible_total", "v_recomp_concentration_indep", "stored_eligible_total", "recomputed_eligible_total")
check_recomp_num("recomp_indep_top_5_travelers", "v_recomp_concentration_indep", "stored_top_5", "recomputed_top_5")
check_recomp_num("recomp_indep_top_10_travelers", "v_recomp_concentration_indep", "stored_top_10", "recomputed_top_10")
check_recomp_num("recomp_indep_top_5_share", "v_recomp_concentration_indep", "stored_top_5_share", "recomputed_top_5_share")
check_recomp_num("recomp_indep_top_10_share", "v_recomp_concentration_indep", "stored_top_10_share", "recomputed_top_10_share")
check_recomp_num("recomp_excluded_count", "v_recomp_concentration_indep", "stored_excluded_count", "recomputed_excluded_count")
check_recomp_num("recomp_top_5_unit_count", "v_recomp_concentration_indep", "stored_top_5_unit_count", "recomputed_top_5_unit_count")
check_recomp_num("recomp_top_10_unit_count", "v_recomp_concentration_indep", "stored_top_10_unit_count", "recomputed_top_10_unit_count")

# Economy
check_recomp_eq("recomp_tourism_change_eligible", "v_recomp_economy", "stored_tourism_change_eligible", "recomp_tourism_change_eligible")
check_recomp_eq("recomp_afs_growth_eligible", "v_recomp_economy", "stored_afs_growth_eligible", "recomp_afs_growth_eligible")
check_recomp_eq("recomp_afs_share_eligible", "v_recomp_economy", "stored_afs_share_eligible", "recomp_afs_share_eligible")
check_recomp_eq("recomp_tourism_intensity_eligible", "v_recomp_economy", "stored_tourism_intensity_eligible", "recomp_tourism_intensity_eligible")
check_recomp_eq("recomp_alignment_eligible", "v_recomp_economy", "stored_alignment_eligible", "recomp_alignment_eligible")
check_recomp_num("recomp_tourism_change_pct", "v_recomp_economy", "stored_change", "recomputed_change")
check_recomp_num("recomp_afs_gva_growth_pct", "v_recomp_economy", "stored_growth", "recomputed_growth")
check_recomp_num("recomp_afs_share_of_gdp", "v_recomp_economy", "stored_share", "recomputed_share")
check_recomp_num("recomp_tourism_intensity_2023", "v_recomp_economy", "stored_intensity", "recomputed_intensity")
check_recomp_eq("recomp_tourism_direction", "v_recomp_economy", "stored_tourism_dir", "recomputed_tourism_dir")
check_recomp_eq("recomp_economy_direction", "v_recomp_economy", "stored_econ_dir", "recomputed_econ_dir")
check_recomp_eq("recomp_tourism_economy_pattern", "v_recomp_economy", "stored_pattern", "recomputed_pattern")
check_recomp_eq("recomp_tourism_economy_pattern_label", "v_recomp_economy", "stored_pattern_label", "recomputed_pattern_label")
check_recomp_eq("recomp_recovery_eligible_2023", "v_recomp_economy", "stored_recovery_eligible_2023", "recomputed_recovery_eligible_2023")
check_recomp_num("recomp_recovery_index_2023", "v_recomp_economy", "stored_recovery_index_2023", "recomputed_recovery_index_2023")
check_recomp_eq("recomp_recovery_class_2023", "v_recomp_economy", "stored_recovery_class_2023", "recomputed_recovery_class_2023")

# Segment recovery
check_recomp_eq("recomp_domestic_eligible", "v_recomp_segment", "stored_domestic_eligible", "recomp_domestic_eligible")
check_recomp_eq("recomp_foreign_eligible", "v_recomp_segment", "stored_foreign_eligible", "recomp_foreign_eligible")
check_recomp_eq("recomp_overseas_filipino_eligible", "v_recomp_segment", "stored_overseas_filipino_eligible", "recomp_overseas_filipino_eligible")
check_recomp_num("recomp_dom_index", "v_recomp_segment", "stored_dom_idx", "recomputed_dom_idx")
check_recomp_eq("recomp_dom_class", "v_recomp_segment", "stored_dom_class", "recomputed_dom_class")
check_recomp_num("recomp_for_index", "v_recomp_segment", "stored_for_idx", "recomputed_for_idx")
check_recomp_eq("recomp_for_class", "v_recomp_segment", "stored_for_class", "recomputed_for_class")
check_recomp_num("recomp_of_index", "v_recomp_segment", "stored_of_idx", "recomputed_of_idx")
check_recomp_eq("recomp_of_class", "v_recomp_segment", "stored_of_class", "recomputed_of_class")

# Segment overall recovery (columns now exist in table and view)
check_recomp_num("recomp_segment_overall_index", "v_recomp_segment", "stored_overall_idx", "recomputed_overall_idx")
check_recomp_eq("recomp_segment_overall_class", "v_recomp_segment", "stored_overall_class", "recomputed_overall_class")

print(f"\nRecomputation checks: {BLOCK_COUNT} total BLOCK checks, {BLOCK_FAILURES} failures, {REVIEW_COUNT} REVIEW findings")

## Phase 3: Reconcile to Gold

Use the Gold versions **stored in the Analytics rows** (not a fresh `DESCRIBE HISTORY`) and read Gold with `VERSION AS OF`. Reconcile persisted base values back to those Gold versions by natural key. Mismatches are BLOCK failures.

In [0]:
%python
# Read stored versions from any Analytics table
sv = spark.sql("SELECT _gold_fact_version, _gold_dim_province_version, _gold_dim_year_version FROM `tuklas_dev`.`05_analytics`.`tourism_activity` LIMIT 1").first()
fact_v = int(sv['_gold_fact_version'])
dim_prov_v = int(sv['_gold_dim_province_version'])
dim_year_v = int(sv['_gold_dim_year_version'])

if not gold_versions_ok:
    print("Phase 3 skipped — Gold version checks failed in Phase 1.")
else:
    spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW v_gold_fact_stored AS SELECT * FROM `tuklas_dev`.`04_gold`.`fact_province_year` VERSION AS OF {fact_v}")
    spark.sql(f"CREATE OR REPLACE TEMPORARY VIEW v_gold_dim_prov_stored AS SELECT * FROM `tuklas_dev`.`04_gold`.`dim_province` VERSION AS OF {dim_prov_v}")
    print(f"  Pinned Gold at stored versions: fact v{fact_v}, dim_province v{dim_prov_v}, dim_year v{dim_year_v}")

    def recon_check(name, sql):
        result = spark.sql(sql).first()
        joined = result['joined_rows']
        expected = result['expected_rows']
        mismatches = result['mismatches']
        assert_block(f"recon_{name}_rowcount", joined == expected, f"joined={joined}, expected={expected}", "recon")
        assert_block(f"recon_{name}_values", mismatches == 0, f"{mismatches} mismatches", "recon")

    recon_check("activity", """
        SELECT COUNT(*) AS joined_rows,
               (SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_activity`) AS expected_rows,
               COALESCE(SUM(CASE WHEN NOT (a.total_travelers <=> f.total_travelers)
                     OR NOT (a.population_total <=> f.population_total)
                     OR NOT (a.population_source <=> f.population_source)
                     OR NOT (a.traveler_value_status <=> f.traveler_value_status)
                     OR NOT (a.comparable_to_2019 <=> f.comparable_to_2019)
                     OR NOT (a.foreign_includes_overseas_filipinos <=> f.foreign_includes_overseas_filipinos)
                     OR NOT (a.derived_values <=> f.derived_values)
                     OR NOT (a.per_capita_recomputed <=> f.per_capita_recomputed)
                     OR NOT (a.province_name <=> d.province_name)
                     OR NOT (a.unit_type <=> d.unit_type)
                     OR NOT (a.region_psgc <=> d.region_psgc)
                     OR NOT (a.region_name <=> d.region_name)
                     OR NOT (a.comparability_note <=> d.comparability_note)
                     THEN 1 ELSE 0 END), 0) AS mismatches
        FROM `tuklas_dev`.`05_analytics`.`tourism_activity` a
        JOIN v_gold_fact_stored f ON a.province_psgc = f.province_psgc AND a.year = f.year
        JOIN v_gold_dim_prov_stored d ON a.province_psgc = d.province_psgc
    """)

    recon_check("recovery", """
        WITH gold_2019 AS (
          SELECT province_psgc, total_travelers AS g_tr_2019, traveler_value_status AS g_st_2019,
                 comparable_to_2019 AS g_comp_2019, foreign_includes_overseas_filipinos AS g_fiof,
                 derived_values AS g_dv_2019, per_capita_recomputed AS g_pcr_2019
          FROM v_gold_fact_stored WHERE year = 2019
        ),
        gold_comp AS (
          SELECT province_psgc, year AS comp_year, total_travelers AS g_tr_comp,
                 traveler_value_status AS g_st_comp, comparable_to_2019 AS g_comp_comp,
                 derived_values AS g_dv_comp, per_capita_recomputed AS g_pcr_comp
          FROM v_gold_fact_stored WHERE year IN (2022, 2023)
        )
        SELECT COUNT(*) AS joined_rows,
               (SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`) AS expected_rows,
               COALESCE(SUM(CASE WHEN NOT (r.travelers_2019 <=> g1.g_tr_2019)
                     OR NOT (r.travelers_comparison_year <=> g2.g_tr_comp)
                     OR NOT (r.status_2019 <=> g1.g_st_2019)
                     OR NOT (r.status_comparison <=> g2.g_st_comp)
                     OR NOT (r.comp_2019 <=> g1.g_comp_2019)
                     OR NOT (r.comp_comparison <=> g2.g_comp_comp)
                     OR NOT (r.foreign_includes_overseas_filipinos <=> g1.g_fiof)
                     OR NOT (r.derived_values_2019 <=> g1.g_dv_2019)
                     OR NOT (r.derived_values_comparison <=> g2.g_dv_comp)
                     OR NOT (r.per_capita_recomputed_2019 <=> g1.g_pcr_2019)
                     OR NOT (r.per_capita_recomputed_comparison <=> g2.g_pcr_comp)
                     OR NOT (r.province_name <=> d.province_name)
                     OR NOT (r.unit_type <=> d.unit_type)
                     OR NOT (r.region_psgc <=> d.region_psgc)
                     OR NOT (r.region_name <=> d.region_name)
                     OR NOT (r.comparability_note <=> d.comparability_note)
                     THEN 1 ELSE 0 END), 0) AS mismatches
        FROM `tuklas_dev`.`05_analytics`.`tourism_recovery` r
        JOIN gold_2019 g1 ON r.province_psgc = g1.province_psgc
        JOIN gold_comp g2 ON r.province_psgc = g2.province_psgc AND r.comparison_year = g2.comp_year
        JOIN v_gold_dim_prov_stored d ON r.province_psgc = d.province_psgc
    """)

    recon_check("segment", """
        WITH gold_2019 AS (
          SELECT province_psgc, total_travelers AS g_tr_2019,
                 domestic_travelers AS g_dom_2019, foreign_travelers AS g_for_2019,
                 overseas_filipinos AS g_of_2019, foreign_includes_overseas_filipinos AS g_fiof,
                 derived_values AS g_dv_2019, per_capita_recomputed AS g_pcr_2019
          FROM v_gold_fact_stored WHERE year = 2019
        ),
        gold_2023 AS (
          SELECT province_psgc, total_travelers AS g_tr_2023, comparable_to_2019 AS g_comp_2023,
                 domestic_travelers AS g_dom_2023, foreign_travelers AS g_for_2023,
                 overseas_filipinos AS g_of_2023, derived_values AS g_dv_2023, per_capita_recomputed AS g_pcr_2023
          FROM v_gold_fact_stored WHERE year = 2023
        )
        SELECT COUNT(*) AS joined_rows,
               (SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`) AS expected_rows,
               COALESCE(SUM(CASE WHEN NOT (s.domestic_travelers_2019 <=> g1.g_dom_2019)
                     OR NOT (s.domestic_travelers_2023 <=> g2.g_dom_2023)
                     OR NOT (s.foreign_travelers_2019 <=> g1.g_for_2019)
                     OR NOT (s.foreign_travelers_2023 <=> g2.g_for_2023)
                     OR NOT (s.overseas_filipino_travelers_2019 <=> g1.g_of_2019)
                     OR NOT (s.overseas_filipino_travelers_2023 <=> g2.g_of_2023)
                     OR NOT (s.travelers_2019 <=> g1.g_tr_2019)
                 OR NOT (s.travelers_2023 <=> g2.g_tr_2023)
                 OR NOT (s.comparable_to_2019 <=> g2.g_comp_2023)
                 OR NOT (s.foreign_includes_overseas_filipinos <=> g1.g_fiof)
                     OR NOT (s.derived_values_2019 <=> g1.g_dv_2019)
                     OR NOT (s.derived_values_2023 <=> g2.g_dv_2023)
                     OR NOT (s.per_capita_recomputed_2019 <=> g1.g_pcr_2019)
                     OR NOT (s.per_capita_recomputed_2023 <=> g2.g_pcr_2023)
                     OR NOT (s.province_name <=> d.province_name)
                     OR NOT (s.unit_type <=> d.unit_type)
                     OR NOT (s.region_psgc <=> d.region_psgc)
                     OR NOT (s.region_name <=> d.region_name)
                     OR NOT (s.comparability_note <=> d.comparability_note)
                     THEN 1 ELSE 0 END), 0) AS mismatches
        FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` s
        JOIN gold_2019 g1 ON s.province_psgc = g1.province_psgc
        JOIN gold_2023 g2 ON s.province_psgc = g2.province_psgc
        JOIN v_gold_dim_prov_stored d ON s.province_psgc = d.province_psgc
    """)

    recon_check("economy", """
        WITH gold_2019 AS (
          SELECT province_psgc, total_travelers AS g_tr_2019, traveler_value_status AS g_st_2019,
                 comparable_to_2019 AS g_comp_2019, afs_gva_constant AS g_afs_2019,
                 gdp_per_capita_current AS g_gdppc_2019, derived_values AS g_dv_2019,
                 per_capita_recomputed AS g_pcr_2019
          FROM v_gold_fact_stored WHERE year = 2019
        ),
        gold_2023 AS (
          SELECT province_psgc, total_travelers AS g_tr_2023, traveler_value_status AS g_st_2023,
                 comparable_to_2019 AS g_comp_2023, afs_gva_constant AS g_afs_2023,
                 afs_gva_current AS g_afs_cur_2023, gdp_total_current AS g_gdp_2023,
                 gdp_per_capita_current AS g_gdppc_2023, population_total AS g_pop_2023,
                 population_source AS g_pop_src_2023, derived_values AS g_dv_2023,
                 per_capita_recomputed AS g_pcr_2023
          FROM v_gold_fact_stored WHERE year = 2023
        )
        SELECT COUNT(*) AS joined_rows,
               (SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_economy`) AS expected_rows,
               COALESCE(SUM(CASE WHEN NOT (e.travelers_2019 <=> g1.g_tr_2019)
                     OR NOT (e.travelers_2023 <=> g2.g_tr_2023)
                     OR NOT (e.status_2019 <=> g1.g_st_2019)
                     OR NOT (e.status_2023 <=> g2.g_st_2023)
                     OR NOT (e.comp_2019 <=> g1.g_comp_2019)
                     OR NOT (e.comp_2023 <=> g2.g_comp_2023)
                     OR NOT (e.afs_gva_constant_2019 <=> g1.g_afs_2019)
                     OR NOT (e.afs_gva_constant_2023 <=> g2.g_afs_2023)
                     OR NOT (e.afs_gva_current_2023 <=> g2.g_afs_cur_2023)
                     OR NOT (e.gdp_total_current_2023 <=> g2.g_gdp_2023)
                     OR NOT (e.gdp_per_capita_current_2019 <=> g1.g_gdppc_2019)
                     OR NOT (e.gdp_per_capita_current_2023 <=> g2.g_gdppc_2023)
                     OR NOT (e.population_total_2023 <=> g2.g_pop_2023)
                     OR NOT (e.population_source_2023 <=> g2.g_pop_src_2023)
                     OR NOT (e.derived_values_2019 <=> g1.g_dv_2019)
                     OR NOT (e.derived_values_2023 <=> g2.g_dv_2023)
                     OR NOT (e.per_capita_recomputed_2019 <=> g1.g_pcr_2019)
                     OR NOT (e.per_capita_recomputed_2023 <=> g2.g_pcr_2023)
                     OR NOT (e.province_name <=> d.province_name)
                     OR NOT (e.unit_type <=> d.unit_type)
                     OR NOT (e.region_psgc <=> d.region_psgc)
                     OR NOT (e.region_name <=> d.region_name)
                     THEN 1 ELSE 0 END), 0) AS mismatches
        FROM `tuklas_dev`.`05_analytics`.`tourism_economy` e
        JOIN gold_2019 g1 ON e.province_psgc = g1.province_psgc
        JOIN gold_2023 g2 ON e.province_psgc = g2.province_psgc
        JOIN v_gold_dim_prov_stored d ON e.province_psgc = d.province_psgc
    """)

    print(f"\nGold reconciliation: {BLOCK_COUNT} total BLOCK checks, {BLOCK_FAILURES} failures")

## Phase 4: Execution summary

Report explicit pass/fail for traceability checks and list any failures.

In [0]:
%python
print("=" * 80)
print("ANALYTICS VALIDATION SUMMARY")
print("=" * 80)

# Traceability info
sv = spark.sql("SELECT _analytics_run_id, _gold_fact_version, _gold_dim_province_version, _gold_dim_year_version FROM `tuklas_dev`.`05_analytics`.`tourism_activity` LIMIT 1").first()
print(f"\nAnalytics run_id: {sv['_analytics_run_id']}")
print(f"Gold fact version: {sv['_gold_fact_version']}")
print(f"Gold dim_province version: {sv['_gold_dim_province_version']}")
print(f"Gold dim_year version: {sv['_gold_dim_year_version']}")

# Per-category pass/fail
print(f"\n--- Per-category results ---")
for cat in ["trace", "recomp", "recon", "rowcount"]:
    if cat in category_counts:
        cc = category_counts[cat]
        status = "PASS" if cc["failures"] == 0 else "FAIL"
        print(f"  [{status}] {cat}: {cc['count'] - cc['failures']}/{cc['count']} passed, {cc['failures']} failed")
    else:
        print(f"  [SKIP] {cat}: 0 checks")

print(f"\nTotal BLOCK checks: {BLOCK_COUNT}")
print(f"BLOCK failures: {BLOCK_FAILURES}")
print(f"REVIEW findings: {REVIEW_COUNT}")

if BLOCK_FAILURES > 0:
    print(f"\nFailures:")
    for item in failures:
        name, details, cat = item
        print(f"  [FAIL] [{cat}] {name}: {details}")

if REVIEW_COUNT > 0:
    print(f"\nReview findings:")
    for name, details in review_findings:
        print(f"  [REVIEW] {name}: {details}")

# Overall result
overall = "NOT READY" if BLOCK_FAILURES > 0 else "READY"

print(f"\n{'=' * 80}")
print(f"OVERALL VALIDATION RESULT: {overall}")
print(f"{'=' * 80}")

if BLOCK_FAILURES > 0:
    raise AssertionError(f"Validation FAILED with {BLOCK_FAILURES} BLOCK failures — see summary above")